In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find Qwen2.5-72B-Instruct project root (paths.py). Run Jupyter with cwd project root or notebooks/."
    )
import paths


In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-72B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=2048,
#     temperature=0.0,
    do_sample=False
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

/home/yuexing/miniconda/envs/openai_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|████████████████████████████████████████████████| 37/37 [01:19<00:00,  2.16s/it]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [3]:
import pandas as pd 
import re
import torch
import os

# Load data
df = pd.read_csv(paths.DATA / "MedGemma_SR_Match_Rate.csv")
print("Columns in dataset:")
print(df.columns.tolist())
print(f"Total rows: {len(df)}")

# Function to extract answer letter using multiple patterns
def extract_answer_letter(text):
    if pd.isna(text) or not text:
        return None
    
    # Try different patterns to extract the answer letter
    patterns = [
        r"Answer:\s*([A-J])",             # "Answer: A"
        r"Answer is\s*([A-J])",           # "Answer is A"
        r"answer is\s*([A-J])",           # "answer is A"
        r"The answer is\s*([A-J])",       # "The answer is A"
        r"the answer is\s*([A-J])",       # "the answer is A"
        r"Option\s*([A-J])",              # "Option A"
        r"option\s*([A-J])",              # "option A"
        r"My answer is\s*([A-J])",        # "My answer is A"
        r"(\n|^)([A-J])\.?\s*$",          # "A." or just "A" at end or newline
        r"select option\s*([A-J])",       # "select option A"
        r"I select\s*([A-J])",            # "I select A"
        r"I choose\s*([A-J])",            # "I choose A"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            # Some patterns have the letter in group 1, others in group 2
            letter = match.group(1) if len(match.groups()) == 1 else match.group(2)
            return letter.upper()
    
    # If no match found, check if there's a single letter at the end
    words = text.strip().split()
    if words and len(words[-1]) == 1 and words[-1].isalpha() and words[-1].upper() in "ABCDEFGHIJ":
        return words[-1].upper()
    
    return None


# Setup progress tracking
progress_file = paths.PREDICTIONS / "Qwen_72B_predictions_on_MedGemma_progress.csv"

# Load existing progress if available
if os.path.exists(progress_file):
    existing_results = pd.read_csv(progress_file)
    # Extract processed Origin IDs
    processed_origins = set(existing_results['Origin'].tolist())
    results = existing_results.to_dict('records')
    print(f"Found {len(processed_origins)} already processed rows. Resuming...")
else:
    processed_origins = set()
    results = []
    print("Starting from scratch...")


# Process each row
total_rows = len(df)
print(f"\nProcessing {total_rows} rows...")

for idx, row in df.iterrows():
    origin_id = row.get("Origin", f"ID{idx:04d}")
    
    # Skip if already processed
    if origin_id in processed_origins:
        print(f"Skipping row {idx+1}/{total_rows} ({origin_id}) - already processed")
        continue
    
    print(f"\nProcessing row {idx+1}/{total_rows} ({origin_id})...")
    
    try:
        # Get context and question
        context_text = row["MedGemma_High_Relevance"]
        question = row["question_options_x"]
        
        # Create prompt
        query_full = (
            "You are a clinical reasoning assistant. You will receive a patient case summary "
            "and a multiple-choice question.\n\n"
            f"{context_text}\n\n"
            f"{question}\n\n"
            "Please select the single most appropriate answer. Respond only in the following format:\n\n"
            "Answer: <LETTER>"
        )
    
        # Generate prediction using the model
        inputs = tokenizer(query_full, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode the generated response
        raw_response = tokenizer.decode(
            output[0][inputs.input_ids.shape[1]:], 
            skip_special_tokens=True
        ).strip()
        
        # Extract the answer letter
        extracted_answer = extract_answer_letter(raw_response)
        
        # Log if extraction failed
        if extracted_answer is None:
            print(f"⚠️  Could not extract answer from response:")
            print(f"   Full Response: '{raw_response}'")
            print(f"   Response length: {len(raw_response)} characters")
        else:
            print(f"✅ Extracted Answer: {extracted_answer}")
        
        # Create result entry
        result_entry = {
            "Origin": origin_id,
            "data_source": row.get("data_source_df3", ""),
            "Raw_Response": raw_response,
            "Extracted_Answer": extracted_answer
        }
        
        results.append(result_entry)
        processed_origins.add(origin_id)
        
        # Save progress every 10 items
        if len(results) % 10 == 0:
            temp_df = pd.DataFrame(results)
            temp_df.to_csv(progress_file, index=False)
            print(f"💾 Saved progress: {len(results)} items processed")

    except Exception as e:
        print(f"❌ Error processing {origin_id}: {str(e)}")
        # Save error entry
        results.append({
            "Origin": origin_id,
            "data_source": row.get("data_source_df3", ""),
            "Raw_Response": f"ERROR: {str(e)}",
            "Extracted_Answer": None
        })
        processed_origins.add(origin_id)

# Save final results
output_df = pd.DataFrame(results)
output_file = paths.PREDICTIONS / "Qwen_72B_predictions_MedGemma.csv"
output_df.to_csv(output_file, index=False)
print(f"\n{'='*60}")
print(f"✅ All predictions saved to {output_file}")
print(f"Total processed: {len(results)} rows")
print(f"{'='*60}")

Columns in dataset:
['Origin', 'data_source_df3', 'Patient_Profile', 'Low+Irr', 'High', 'question_options_x', 'answer_corr', 'ID', 'centaur_question', 'sentence_number', 'answer', 'data_source', 'step1_excerpts', 'question_options_y', 'step1_sentences', 'sentence_1', 'sentence_2', 'sentence_3', 'sentence_4', 'sentence_5', 'sentence_6', 'sentence_7', 'sentence_8', 'sentence_9', 'sentence_10', 'sentence_11', 'sentence_12', 'sentence_13', 'sentence_14', 'sentence_15', 'sentence_16', 'sentence_17', 'sentence_18', 'sentence_19', 'sentence_20', 'sentence_21', 'non_none_sentence_count', 'MedGemma27B_answer', 'MedGemma27B_raw_response', 'q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10', 'q11', 'q12', 'q13', 'q14', 'q15', 'q16', 'q17', 'q18', 'q19', 'q20', 'label_21', 'MedGemma_High_Relevance', 'Match?', 'data_source_corr_trainee', 'Match_percent']
Total rows: 1300
Found 120 already processed rows. Resuming...

Processing 1300 rows...
Skipping row 1/1300 (ID0002) - already processed
S

✅ Extracted Answer: B

Processing row 141/1300 (ID0233)...
✅ Extracted Answer: B

Processing row 142/1300 (ID0235)...
✅ Extracted Answer: H

Processing row 143/1300 (ID0236)...
✅ Extracted Answer: C

Processing row 144/1300 (ID0237)...
✅ Extracted Answer: A

Processing row 145/1300 (ID0238)...
✅ Extracted Answer: D

Processing row 146/1300 (ID0240)...
✅ Extracted Answer: C

Processing row 147/1300 (ID0241)...
✅ Extracted Answer: A

Processing row 148/1300 (ID0242)...
✅ Extracted Answer: G

Processing row 149/1300 (ID0243)...
✅ Extracted Answer: C
💾 Saved progress: 150 items processed

Processing row 150/1300 (ID0244)...
✅ Extracted Answer: C

Processing row 151/1300 (ID0246)...
✅ Extracted Answer: D

Processing row 152/1300 (ID0247)...
✅ Extracted Answer: C

Processing row 153/1300 (ID0248)...
✅ Extracted Answer: A

Processing row 154/1300 (ID0249)...
✅ Extracted Answer: C

Processing row 155/1300 (ID0251)...
✅ Extracted Answer: C

Processing row 156/1300 (ID0253)...
✅ Extracted Answer

✅ Extracted Answer: B

Processing row 246/1300 (ID0404)...
✅ Extracted Answer: A

Processing row 247/1300 (ID0405)...
✅ Extracted Answer: B

Processing row 248/1300 (ID0406)...
⚠️  Could not extract answer from response:
   Full Response: 'To ensure you receive the correct response, please provide the patient's age, gender, and any other relevant medical history or symptoms.
To provide the most accurate response, I need additional information about the patient, such as age, gender, and any other relevant medical history or symptoms. Could you please provide these details? 

However, based on the information provided, the patient appears to be experiencing symptoms of depression, including depressed mood, constricted affect, loss of interest in activities, and suicidal ideation'
   Response length: 547 characters

Processing row 249/1300 (ID0407)...
✅ Extracted Answer: A
💾 Saved progress: 250 items processed

Processing row 250/1300 (ID0409)...
⚠️  Could not extract answer from response

✅ Extracted Answer: C

Processing row 358/1300 (ID0585)...
⚠️  Could not extract answer from response:
   Full Response: 'To provide the best possible care, further details about the patient's condition would be helpful. Could you please share the patient case summary? Answer: N
To provide the best possible care, further details about the patient's condition would be helpful. Could you please share the patient case summary? 

Answer: N'
   Response length: 317 characters

Processing row 359/1300 (ID0586)...
✅ Extracted Answer: B
💾 Saved progress: 360 items processed

Processing row 360/1300 (ID0587)...
✅ Extracted Answer: E

Processing row 361/1300 (ID0589)...
✅ Extracted Answer: C

Processing row 362/1300 (ID0590)...
✅ Extracted Answer: D

Processing row 363/1300 (ID0591)...
✅ Extracted Answer: G

Processing row 364/1300 (ID0592)...
✅ Extracted Answer: C

Processing row 365/1300 (ID0594)...
✅ Extracted Answer: G

Processing row 366/1300 (ID0595)...
✅ Extracted Answer: J

Processing ro

✅ Extracted Answer: D

Processing row 464/1300 (ID0743)...
✅ Extracted Answer: B

Processing row 465/1300 (ID0744)...
✅ Extracted Answer: D

Processing row 466/1300 (ID0745)...
✅ Extracted Answer: D

Processing row 467/1300 (ID0746)...
✅ Extracted Answer: C

Processing row 468/1300 (ID0747)...
✅ Extracted Answer: A

Processing row 469/1300 (ID0752)...
✅ Extracted Answer: B
💾 Saved progress: 470 items processed

Processing row 470/1300 (ID0753)...
✅ Extracted Answer: A

Processing row 471/1300 (ID0754)...
✅ Extracted Answer: B

Processing row 472/1300 (ID0756)...
✅ Extracted Answer: B

Processing row 473/1300 (ID0757)...
✅ Extracted Answer: B

Processing row 474/1300 (ID0758)...
✅ Extracted Answer: D

Processing row 475/1300 (ID0759)...
✅ Extracted Answer: B

Processing row 476/1300 (ID0761)...
✅ Extracted Answer: D

Processing row 477/1300 (ID0762)...
✅ Extracted Answer: C

Processing row 478/1300 (ID0764)...
⚠️  Could not extract answer from response:
   Full Response: 'To address the

...

✅ Extracted Answer: B

Processing row 916/1300 (ID1418)...
✅ Extracted Answer: D

Processing row 917/1300 (ID1419)...
✅ Extracted Answer: B

Processing row 918/1300 (ID1421)...
✅ Extracted Answer: D

Processing row 919/1300 (ID1422)...
✅ Extracted Answer: B
💾 Saved progress: 920 items processed

Processing row 920/1300 (ID1423)...
✅ Extracted Answer: B

Processing row 921/1300 (ID1425)...
⚠️  Could not extract answer from response:
   Full Response: 'To provide the best care for this patient, we need to consider her medical history, including her HIV status and previous adverse reactions to antibiotics. The patient has a history of anaphylaxis to penicillin, which rules out amoxicillin (A). She also has a history of a rash with erythromycin, which is in the same class as clarithromycin (C), making it less ideal due to potential cross-reactivity.

Given the symptoms of a swollen, eryth'
   Response length: 442 characters

Processing row 922/1300 (ID1426)...
✅ Extracted Answer: D

Proces

✅ Extracted Answer: A

Processing row 1013/1300 (ID1569)...
✅ Extracted Answer: B

Processing row 1014/1300 (ID1570)...
✅ Extracted Answer: A

Processing row 1015/1300 (ID1571)...
✅ Extracted Answer: C

Processing row 1016/1300 (ID1572)...
✅ Extracted Answer: G

Processing row 1017/1300 (ID1573)...
✅ Extracted Answer: C

Processing row 1018/1300 (ID1574)...
✅ Extracted Answer: C

Processing row 1019/1300 (ID1575)...
✅ Extracted Answer: C
💾 Saved progress: 1020 items processed

Processing row 1020/1300 (ID1578)...
✅ Extracted Answer: G

Processing row 1021/1300 (ID1580)...
✅ Extracted Answer: J

Processing row 1022/1300 (ID1581)...
✅ Extracted Answer: D

Processing row 1023/1300 (ID1582)...
✅ Extracted Answer: J

Processing row 1024/1300 (ID1583)...
⚠️  Could not extract answer from response:
   Full Response: 'To provide the correct answer, let's analyze the information given:

1. The patient has a linear, nondepressed basal skull fracture.
2. Two weeks later, the patient develops poly

✅ Extracted Answer: D

Processing row 1081/1300 (ID1680)...
✅ Extracted Answer: C

Processing row 1082/1300 (ID1683)...
✅ Extracted Answer: C

Processing row 1083/1300 (ID1684)...
✅ Extracted Answer: A

Processing row 1084/1300 (ID1685)...
✅ Extracted Answer: B

Processing row 1085/1300 (ID1687)...
✅ Extracted Answer: C

Processing row 1086/1300 (ID1688)...
✅ Extracted Answer: D

Processing row 1087/1300 (ID1689)...
✅ Extracted Answer: A

Processing row 1088/1300 (ID1691)...
✅ Extracted Answer: J

Processing row 1089/1300 (ID1692)...
✅ Extracted Answer: C
💾 Saved progress: 1090 items processed

Processing row 1090/1300 (ID1693)...
✅ Extracted Answer: C

Processing row 1091/1300 (ID1694)...
✅ Extracted Answer: J

Processing row 1092/1300 (ID1695)...
✅ Extracted Answer: B

Processing row 1093/1300 (ID1696)...
✅ Extracted Answer: F

Processing row 1094/1300 (ID1697)...
✅ Extracted Answer: B

Processing row 1095/1300 (ID1698)...
✅ Extracted Answer: G

Processing row 1096/1300 (ID1699)...
✅

⚠️  Could not extract answer from response:
   Full Response: 'To provide a detailed explanation, please ask me to elaborate.
To determine the most likely spinal level demonstrating a sympathetic viscerosomatic reflex associated with the patient's condition, we need to consider the anatomical innervation of the lower abdomen and the colon.

The lower abdomen and the descending colon are primarily innervated by the sympathetic fibers that originate from the thoracolumbar region of the spinal cord, specifically from T10 to L2. The most common level associated with the lower abdomen and'
   Response length: 527 characters

Processing row 1183/1300 (ID1835)...
✅ Extracted Answer: B

Processing row 1184/1300 (ID1836)...
✅ Extracted Answer: C

Processing row 1185/1300 (ID1837)...
✅ Extracted Answer: C

Processing row 1186/1300 (ID1838)...
✅ Extracted Answer: B

Processing row 1187/1300 (ID1840)...
✅ Extracted Answer: D

Processing row 1188/1300 (ID1841)...
✅ Extracted Answer: A

Processing r

✅ Extracted Answer: C

Processing row 1265/1300 (ID1951)...
✅ Extracted Answer: C

Processing row 1266/1300 (ID1952)...
✅ Extracted Answer: A

Processing row 1267/1300 (ID1953)...
✅ Extracted Answer: B

Processing row 1268/1300 (ID1955)...
✅ Extracted Answer: A

Processing row 1269/1300 (ID1956)...
✅ Extracted Answer: C
💾 Saved progress: 1270 items processed

Processing row 1270/1300 (ID1957)...
✅ Extracted Answer: E

Processing row 1271/1300 (ID1958)...
✅ Extracted Answer: B

Processing row 1272/1300 (ID1959)...
✅ Extracted Answer: D

Processing row 1273/1300 (ID1962)...
✅ Extracted Answer: B

Processing row 1274/1300 (ID1963)...
✅ Extracted Answer: B

Processing row 1275/1300 (ID1965)...
✅ Extracted Answer: C

Processing row 1276/1300 (ID1966)...
✅ Extracted Answer: A

Processing row 1277/1300 (ID1969)...
✅ Extracted Answer: A

Processing row 1278/1300 (ID1970)...
✅ Extracted Answer: B

Processing row 1279/1300 (ID1972)...
✅ Extracted Answer: C
💾 Saved progress: 1280 items processed


In [4]:
import pandas as pd
import re
import numpy as np
from scipy import stats

# Load the model predictions
output_df = pd.read_csv(paths.PREDICTIONS / "Qwen_72B_predictions_MedGemma.csv")
df = pd.read_csv(paths.DATA / "After_Removal_High_qwen_72B_predictions.csv")

# Merge the two dataframes on ID_corr
merged_df = pd.merge(df, output_df[['Origin', 'Extracted_Answer']], on='Origin', how='inner')

# Compare answers
merged_df['72B_on_70B_Match'] = merged_df['answer_df3'] == merged_df['Extracted_Answer']
merged_df['72B_on_70B_Match'] = merged_df['72B_on_70B_Match'].map({True: 'TRUE', False: 'FALSE'})

# Exact match as 0/1 for statistics
merged_df['match'] = (merged_df['answer_df3'] == merged_df['Extracted_Answer']).astype(int)

# Overall accuracy statistics
accuracy = merged_df['match'].mean()
std_dev = merged_df['match'].std()
n = len(merged_df)
se = std_dev / np.sqrt(n)
ci_95 = stats.t.interval(0.95, n-1, loc=accuracy, scale=se)

print("=" * 60)
print("OVERALL ACCURACY ANALYSIS")
print("=" * 60)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Standard Deviation: {std_dev:.4f}")
print(f"95% Confidence Interval: [{ci_95[0]:.4f}, {ci_95[1]:.4f}]")
print(f"95% CI (percentage): [{ci_95[0]*100:.2f}%, {ci_95[1]*100:.2f}%]")
print(f"Sample Size: {n}")
print("=" * 60)
print()

# Category-wise analysis (if data_source_corr exists in df)
if 'data_source_corr' in df.columns:
    merged_df['data_source_corr'] = merged_df['data_source_corr']
    category_stats = merged_df.groupby('data_source_corr')['match'].agg([
        ('Count', 'count'),
        ('Mean_Accuracy', 'mean'),
        ('Std_Dev', 'std'),
        ('SE', lambda x: x.std() / np.sqrt(len(x)))
    ]).reset_index()

    # Calculate 95% CI per category
    ci_lower, ci_upper = [], []
    for idx, row in category_stats.iterrows():
        n_cat = row['Count']
        mean_cat = row['Mean_Accuracy']
        se_cat = row['SE']
        if n_cat > 1:
            ci = stats.t.interval(0.95, n_cat-1, loc=mean_cat, scale=se_cat)
            ci_lower.append(ci[0])
            ci_upper.append(ci[1])
        else:
            ci_lower.append(np.nan)
            ci_upper.append(np.nan)
    
    category_stats['CI_95_Lower'] = ci_lower
    category_stats['CI_95_Upper'] = ci_upper
    category_stats['Mean_Accuracy_%'] = category_stats['Mean_Accuracy'] * 100
    category_stats['Std_Dev_%'] = category_stats['Std_Dev'] * 100
    category_stats['CI_95_Lower_%'] = category_stats['CI_95_Lower'] * 100
    category_stats['CI_95_Upper_%'] = category_stats['CI_95_Upper'] * 100

    print("CATEGORY-WISE ACCURACY ANALYSIS")
    print("=" * 60)
    print(category_stats.to_string(index=False))
    print("=" * 60)
    print()

# Summary table
summary_table = pd.DataFrame({
    'Metric': ['Overall Accuracy', 'Standard Deviation', '95% CI Lower', '95% CI Upper', 'Sample Size'],
    'Value': [f"{accuracy:.4f} ({accuracy*100:.2f}%)", 
              f"{std_dev:.4f}", 
              f"{ci_95[0]:.4f} ({ci_95[0]*100:.2f}%)", 
              f"{ci_95[1]:.4f} ({ci_95[1]*100:.2f}%)", 
              n]
})

print("\nSUMMARY TABLE")
print("=" * 60)
print(summary_table.to_string(index=False))
print("=" * 60)

OVERALL ACCURACY ANALYSIS
Accuracy: 0.5878 (58.78%)
Standard Deviation: 0.4924
95% Confidence Interval: [0.5610, 0.6146]
95% CI (percentage): [56.10%, 61.46%]
Sample Size: 1298

CATEGORY-WISE ACCURACY ANALYSIS
data_source_corr  Count  Mean_Accuracy  Std_Dev       SE  CI_95_Lower  CI_95_Upper  Mean_Accuracy_%  Std_Dev_%  CI_95_Lower_%  CI_95_Upper_%
            jama    583       0.679245 0.467168 0.019348     0.641245     0.717246        67.924528  46.716750      64.124468      71.724589
      medbullets    207       0.642512 0.480422 0.033392     0.576679     0.708345        64.251208  48.042201      57.667890      70.834525
        medxpert    315       0.238095 0.426595 0.024036     0.190803     0.285387        23.809524  42.659538      19.080341      28.538707
            mmlu    193       0.823834 0.381952 0.027494     0.769606     0.878062        82.383420  38.195197      76.960611      87.806229


SUMMARY TABLE
            Metric           Value
  Overall Accuracy 0.5878 (58.78%)